# Revision 01 Fit Four-Condition Ratio-Matched MS-AA


Targeted revision analysis adding the **paragraph-scrambled** condition.

This notebook fits **within-condition** MS-AA models for intact, paragraph-scrambled,
word-scrambled, and rest. This avoids requiring equal temporal extent across
conditions while still allowing the spatial and temporal formulations to be
compared at matched *relative* model complexity.

For each requested normalized ratio \(r\):

- spatial AA uses \(K \approx r \times 700\);
- temporal AA uses \(K \approx r \times T_c\), where \(T_c\) is the number of
  timepoints available in condition \(c\).

Thus the two orientations are compared at approximately equal fractions of their
available decomposition dimensions.

The notebook intentionally saves into a **new output directory** and does not alter
the original across-condition fits.


## Paragraph duplicate handling

The source paragraph-scrambled array contains a duplicated subject entry. Before any
normalization, fitting, PCA, raw-HTFA decoding, or MS-AA decoding, the duplicate is
removed using the same parsing rule as the original analysis:

- **700-factor HTFA:** exclude paragraph source-column index **3**;
- **other factor counts:** exclude paragraph source-column index **0**.

The retained source indices and resulting participant counts are printed explicitly so
that the four-condition pipeline can be audited.

In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================

ANALYSIS_TYPES = ["spatial", "temporal"]
CONDITIONS = ["intact", "paragraph", "word", "rest"]

# HTFA dimensionality used for this analysis.
# Controls which duplicated paragraph entry is removed.
FACTORS = 700

# Match the parsing used in the original analysis:
#   factors == 700 -> exclude paragraph column index 3
#   otherwise      -> exclude paragraph column index 0
PARAGRAPH_DUPLICATE_INDEX_BY_FACTORS = {
    700: 3,
    "default": 0,
}


# A reasonably dense but manageable ratio grid.
# Add/remove values if desired.
TARGET_RATIOS = [
    0.01, 0.02, 0.03, 0.05, 0.075,
    0.10, 0.125, 0.15, 0.20, 0.25,
    0.30, 0.40, 0.50, 0.75, 1.00
]

# Optional chunking if you do not want to launch all fits at once.
# Example: RATIO_INDICES_TO_RUN = list(range(0,5))
RATIO_INDICES_TO_RUN = None

TRIM_REST_BY = 100
RNG_SEED = 42

PIEMAN_MAT_PATH = "data/input/pieman_data.mat"
POSTERIOR_MAT_PATH = "data/input/pieman_posterior_K700.mat"
HELPERS_DIR = "helpers"
HYPERTOOLS_NORMALIZE_PATH = "hypertools/normalize.py"

OUTPUT_DIR = "msaa_four_condition_ratio_within_outputs_npz"

MSAA_OPTS = dict(
    maxiter=200,
    conv_crit=1e-6,
    fix_var_iter=5,
    heteroscedastic=False,
    use_gpu=False,
    rngSEED=RNG_SEED,
)

SKIP_EXISTING = True
STOP_ON_ERROR = False
VERBOSE_PROGRESS = True

# Candidate MAT-file aliases. The resolver below prints the actual key used.
CONDITION_KEY_ALIASES = {
    "intact": ["intact"],
    "paragraph": [
        "paragraph", "paragraph_scrambled", "paragraph-scrambled",
        "para", "paragraphscrambled"
    ],
    "word": ["word", "word_scrambled", "word-scrambled", "wordscrambled"],
    "rest": ["rest"],
}

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

%matplotlib inline

import os
import sys
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat

HT_NORMALIZE_AVAILABLE = False

try:
    spec = importlib.util.spec_from_file_location(
        "normalize",
        HYPERTOOLS_NORMALIZE_PATH,
    )
    ht = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ht)
    HT_NORMALIZE_AVAILABLE = True
except Exception:
    ht = None

sys.path.append(HELPERS_DIR)
from MultiSubject_AA import Subject, multi_subject_aa

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("Ready.")

In [ ]:
# ============================================================
# DATA HELPERS
# ============================================================

def to_float_array(x):
    return np.asarray(x, dtype=float)

def array_cutter(data_list, cut_length):
    out = [to_float_array(x).copy() for x in data_list]
    return out if cut_length <= 0 else [x[:-cut_length] for x in out]

def coerce_TxV(x):
    x = to_float_array(x)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D array, got {x.shape}")

    T0, V0 = x.shape

    # Same orientation convention as the original fitting notebook.
    if V0 < T0:
        x = x.T

    x = x - x.mean(axis=0, keepdims=True)

    return x.astype(np.float32, copy=False)

def normalize_across_fallback(subject_list, eps=1e-8):
    subj = [coerce_TxV(x) for x in subject_list]
    stacked = np.vstack(subj)

    mu = stacked.mean(axis=0, keepdims=True)
    sd = stacked.std(axis=0, keepdims=True) + eps

    return [
        ((x - mu) / sd).astype(np.float32, copy=False)
        for x in subj
    ]

def normalize_subject_list(subject_list):
    subj = [coerce_TxV(x) for x in subject_list]

    if HT_NORMALIZE_AVAILABLE:
        try:
            return ht.normalize(subj, normalize="across")
        except Exception:
            pass

    return normalize_across_fallback(subj)

def resolve_condition_key(mat, canonical_name):
    """
    Resolve paragraph/word aliases without assuming the exact key in the .mat file.
    """
    available = [k for k in mat.keys() if not k.startswith("__")]
    lower_map = {str(k).lower(): k for k in available}

    for alias in CONDITION_KEY_ALIASES[canonical_name]:
        if alias.lower() in lower_map:
            return lower_map[alias.lower()]

    # relaxed normalization
    def norm(s):
        return (
            str(s).lower()
            .replace("_", "")
            .replace("-", "")
            .replace(" ", "")
        )

    normalized = {norm(k): k for k in available}

    for alias in CONDITION_KEY_ALIASES[canonical_name]:
        if norm(alias) in normalized:
            return normalized[norm(alias)]

    raise KeyError(
        f"Could not resolve condition '{canonical_name}'. "
        f"Available MAT keys: {available}"
    )

def paragraph_duplicate_index(factors=FACTORS):
    """Return the duplicated paragraph source-column index to remove."""
    return int(
        PARAGRAPH_DUPLICATE_INDEX_BY_FACTORS.get(
            factors,
            PARAGRAPH_DUPLICATE_INDEX_BY_FACTORS["default"],
        )
    )

def extract_condition_subjects(mat, key, condition, factors=FACTORS):
    """Extract subjects, removing the duplicate paragraph entry before normalization."""
    arr = mat[key]
    all_indices = np.arange(arr.shape[1])

    if condition == "paragraph":
        duplicate_idx = paragraph_duplicate_index(factors)
        if duplicate_idx < 0 or duplicate_idx >= arr.shape[1]:
            raise IndexError(
                f"Paragraph duplicate index {duplicate_idx} is invalid for "
                f"{arr.shape[1]} paragraph entries."
            )
        keep_indices = all_indices[all_indices != duplicate_idx]
        print(
            f"Paragraph parsing: factors={factors}; excluding source column "
            f"index {duplicate_idx}; keeping {len(keep_indices)} of "
            f"{len(all_indices)} entries."
        )
    else:
        keep_indices = all_indices

    subjects = [arr[:, i][0] for i in keep_indices]
    return subjects, keep_indices

def load_condition_data():
    mat = loadmat(PIEMAN_MAT_PATH)

    print("Available MAT variables:")
    print([k for k in mat.keys() if not k.startswith("__")])

    condition_data = {}
    resolved_keys = {}
    retained_source_indices = {}

    for condition in CONDITIONS:
        key = resolve_condition_key(mat, condition)
        resolved_keys[condition] = key

        subjects, keep_indices = extract_condition_subjects(
            mat, key, condition, factors=FACTORS
        )
        retained_source_indices[condition] = keep_indices

        if condition == "rest" and TRIM_REST_BY > 0:
            subjects = array_cutter(subjects, TRIM_REST_BY)

        condition_data[condition] = normalize_subject_list(subjects)

    return condition_data, resolved_keys, retained_source_indices

condition_data, resolved_keys, retained_source_indices = load_condition_data()

print("\nResolved condition keys:")
print(resolved_keys)

print("\nRetained source indices:")
for c in CONDITIONS:
    print(c, retained_source_indices[c])

condition_meta_rows = []

for condition in CONDITIONS:
    X0 = coerce_TxV(condition_data[condition][0])
    T_c, V_c = X0.shape

    condition_meta_rows.append({
        "condition": condition,
        "mat_key": resolved_keys[condition],
        "N": len(condition_data[condition]),
        "T": T_c,
        "V": V_c,
        "chance": 1.0 / T_c,
        "excluded_source_index": (
            paragraph_duplicate_index(FACTORS)
            if condition == "paragraph"
            else np.nan
        ),
        "retained_source_indices": ",".join(
            map(str, retained_source_indices[condition])
        ),
    })

condition_meta = pd.DataFrame(condition_meta_rows)
display(condition_meta)

if not np.all(condition_meta["V"] == FACTORS):
    raise ValueError(
        f"Expected the HTFA representation to contain V={FACTORS} nodes in every condition."
    )

condition_meta.to_csv(
    Path(OUTPUT_DIR) / "four_condition_metadata.csv",
    index=False,
)


## Ratio-matched \(K\) values

The ratio grid is converted separately for each condition because temporal AA's
available decomposition dimension is the condition's own \(T_c\). Spatial AA
always uses \(D=700\).

The table below is the authoritative map used by the fitting, decoding, and
figure notebooks.

In [ ]:
# ============================================================
# BUILD RATIO-MATCHED K TABLE
# ============================================================

def k_from_ratio(r, denominator):
    return int(np.clip(np.rint(r * denominator), 1, denominator))

ratio_rows = []

for condition in CONDITIONS:
    T_c = int(
        condition_meta.loc[
            condition_meta["condition"] == condition,
            "T"
        ].iloc[0]
    )

    for r in TARGET_RATIOS:
        K_spatial = k_from_ratio(r, 700)
        K_temporal = k_from_ratio(r, T_c)

        ratio_rows.append({
            "condition": condition,
            "target_ratio": float(r),
            "analysis_type": "spatial",
            "denominator": 700,
            "K": K_spatial,
            "actual_ratio": K_spatial / 700,
        })

        ratio_rows.append({
            "condition": condition,
            "target_ratio": float(r),
            "analysis_type": "temporal",
            "denominator": T_c,
            "K": K_temporal,
            "actual_ratio": K_temporal / T_c,
        })

ratio_table = pd.DataFrame(ratio_rows)

# Multiple target ratios can round to the same integer K at very low T.
# Keep the first requested target for actual fitting, but preserve full table.
fit_ratio_table = (
    ratio_table
    .sort_values(["analysis_type", "condition", "target_ratio"])
    .drop_duplicates(["analysis_type", "condition", "K"])
    .reset_index(drop=True)
)

ratio_table.to_csv(
    Path(OUTPUT_DIR) / "ratio_K_map_all_requested.csv",
    index=False,
)

fit_ratio_table.to_csv(
    Path(OUTPUT_DIR) / "ratio_K_map_unique_fits.csv",
    index=False,
)

display(fit_ratio_table)

print("\nUnique fits by analysis / condition:")
display(
    fit_ratio_table
    .groupby(["analysis_type", "condition"])
    .size()
    .reset_index(name="n_fits")
)

In [ ]:
# ============================================================
# MS-AA HELPERS
# ============================================================

posterior = loadmat(POSTERIOR_MAT_PATH)
centers = to_float_array(
    posterior["posterior"]["centers"][0][0][0][0][0]
)
widths = to_float_array(
    list(
        posterior["posterior"]["widths"][0][0][0][0][0][:, 0].T
    )
).ravel()

def orient_subject_matrix(X_raw, analysis_type):
    Xtv = coerce_TxV(X_raw)

    if analysis_type == "spatial":
        return Xtv

    if analysis_type == "temporal":
        return Xtv.T

    raise ValueError("analysis_type must be spatial or temporal")

def build_subject_objects(subject_list, analysis_type):
    return [
        Subject(
            X=orient_subject_matrix(x, analysis_type),
            sX=orient_subject_matrix(x, analysis_type),
        )
        for x in subject_list
    ]

def expected_shape_string(example_X_raw, analysis_type, K):
    Xtv = coerce_TxV(example_X_raw)
    T_c, V_c = Xtv.shape

    if analysis_type == "spatial":
        return f"sXC ~ ({T_c}, {K}), S ~ ({K}, {V_c})"

    return f"sXC ~ ({V_c}, {K}), S ~ ({K}, {T_c})"

def fit_msaa(subject_list, K, analysis_type, opts=None, verbose=True):
    opts = dict(MSAA_OPTS if opts is None else opts)

    subj_objs = build_subject_objects(
        subject_list,
        analysis_type,
    )

    results_subj, C, cost_fun, varexpl, secs = multi_subject_aa(
        subj_objs,
        noc=K,
        opts=opts,
    )

    if verbose:
        print(
            f"[{analysis_type.upper()} AA] K={K} | "
            f"VarExpl={varexpl*100:.2f}%"
        )
        print(
            "subj0 sXC:",
            np.asarray(results_subj[0]["sXC"]).shape,
        )
        print(
            "subj0 S:",
            np.asarray(results_subj[0]["S"]).shape,
        )
        print(
            "Expected:",
            expected_shape_string(
                subject_list[0],
                analysis_type,
                K,
            ),
        )

    return {
        "results_subj": results_subj,
        "C": C,
        "cost_fun": cost_fun,
        "varexpl": varexpl,
        "secs": secs,
        "K": K,
    }

def output_path(analysis_type, condition, K):
    return (
        Path(OUTPUT_DIR)
        / f"{analysis_type}AA_within_{condition}_K{K}.npz"
    )

def save_fit_npz(
    save_path,
    fit_res,
    condition,
    target_ratio,
    actual_ratio,
    denominator,
):
    np.savez_compressed(
        save_path,
        K=np.array(fit_res["K"]),
        results_subj=np.array(
            fit_res["results_subj"],
            dtype=object,
        ),
        C=np.array(
            fit_res["C"],
            dtype=object,
        ),
        cost_fun=np.array(
            fit_res["cost_fun"],
            dtype=object,
        ),
        varexpl=np.array(
            fit_res["varexpl"]
        ),
        secs=np.array(
            fit_res["secs"]
        ),
        condition_labels_str=np.array(
            [condition] * len(fit_res["results_subj"]),
            dtype=object,
        ),
        condition_names=np.array(
            [condition],
            dtype=object,
        ),
        analysis_type=np.array(
            analysis_type_current,
            dtype=object,
        ),
        fit_scope=np.array(
            "within",
            dtype=object,
        ),
        target_ratio=np.array(
            target_ratio,
            dtype=float,
        ),
        actual_ratio=np.array(
            actual_ratio,
            dtype=float,
        ),
        ratio_denominator=np.array(
            denominator,
            dtype=int,
        ),
        centers=np.array(centers),
        widths=np.array(widths),
    )

In [ ]:
# ============================================================
# RUN FITS
# ============================================================

rows_to_run = fit_ratio_table.copy()

if RATIO_INDICES_TO_RUN is not None:
    requested = set(
        float(TARGET_RATIOS[i])
        for i in RATIO_INDICES_TO_RUN
    )

    rows_to_run = rows_to_run[
        rows_to_run["target_ratio"].isin(requested)
    ].copy()

saved_paths = []
failed_jobs = []

for _, row in rows_to_run.iterrows():

    analysis_type_current = str(row["analysis_type"])
    condition = str(row["condition"])
    K = int(row["K"])
    target_ratio = float(row["target_ratio"])
    actual_ratio = float(row["actual_ratio"])
    denominator = int(row["denominator"])

    save_path = output_path(
        analysis_type_current,
        condition,
        K,
    )

    print("\n" + "#" * 90)
    print(
        f"{analysis_type_current} | {condition} | "
        f"target ratio={target_ratio:.4f} | "
        f"K={K}/{denominator}={actual_ratio:.4f}"
    )
    print("#" * 90)

    if SKIP_EXISTING and save_path.exists():
        print("Skipping existing:", save_path)
        saved_paths.append(str(save_path))
        continue

    try:
        fit_res = fit_msaa(
            condition_data[condition],
            K,
            analysis_type_current,
            opts=MSAA_OPTS,
            verbose=VERBOSE_PROGRESS,
        )

        save_fit_npz(
            save_path,
            fit_res,
            condition,
            target_ratio,
            actual_ratio,
            denominator,
        )

        saved_paths.append(str(save_path))

        print("Saved:", save_path)

    except Exception as e:
        failed_jobs.append({
            "analysis_type": analysis_type_current,
            "condition": condition,
            "K": K,
            "target_ratio": target_ratio,
            "error_type": type(e).__name__,
            "error_message": str(e),
        })

        print(
            f"FAILED: {analysis_type_current} {condition} K={K}: "
            f"{type(e).__name__}: {e}"
        )

        if STOP_ON_ERROR:
            raise

print("\nSaved:", len(saved_paths))
print("Failed:", len(failed_jobs))

if failed_jobs:
    display(pd.DataFrame(failed_jobs))

In [ ]:
# ============================================================
# AUDIT COMPLETENESS
# ============================================================

audit = rows_to_run.copy()

audit["path"] = audit.apply(
    lambda r: str(
        output_path(
            str(r["analysis_type"]),
            str(r["condition"]),
            int(r["K"]),
        )
    ),
    axis=1,
)

audit["exists"] = audit["path"].map(
    lambda p: Path(p).exists()
)

display(
    audit.sort_values(
        ["analysis_type", "condition", "target_ratio"]
    )
)

print(
    audit.groupby(
        ["analysis_type", "condition"]
    )["exists"].agg(["sum", "count"])
)

audit.to_csv(
    Path(OUTPUT_DIR) / "fit_audit.csv",
    index=False,
)